In [13]:
import os
import random
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Фіксація випадкових значень для відтворюваності результатів
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# Перевірка доступності апаратного прискорювача
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Використовується пристрій: {device}")

Використовується пристрій: cpu


In [3]:
import os
from sklearn.model_selection import train_test_split

# 1. Визначення шляхів до папок з урахуванням структури у каталозі datasets
base_train = os.path.join('datasets', 'seg_train')
base_test = os.path.join('datasets', 'seg_test')

# Перевірка наявності вкладеної папки seg_train/seg_train (якщо розпаковано з підпапкою)
train_dir = os.path.join(base_train, 'seg_train') if os.path.exists(os.path.join(base_train, 'seg_train')) else base_train
test_dir = os.path.join(base_test, 'seg_test') if os.path.exists(os.path.join(base_test, 'seg_test')) else base_test

print(f"Шлях до тренувальних даних: {train_dir}")
print(f"Шлях до тестових даних:      {test_dir}")

# 2. Визначення трансформацій
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_test_transforms = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# 3. Завантаження датасетів через ImageFolder
full_train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
val_base_dataset = datasets.ImageFolder(root=train_dir, transform=val_test_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=val_test_transforms)

# Стратифіковане розбиття тренувальних даних на Train (80%) та Validation (20%)
targets = full_train_dataset.targets
train_idx, val_idx = train_test_split(
    list(range(len(targets))),
    test_size=0.2,
    stratify=targets,
    random_state=42
)

train_subset = Subset(full_train_dataset, train_idx)
val_subset = Subset(val_base_dataset, val_idx)

# 4. Створення DataLoader (для CPU залишаємо num_workers=0)
batch_size = 32

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

# 5. Вивід інформації про вибірки та батчі
class_names = full_train_dataset.classes
print(f"\nКількість класів: {len(class_names)}")
print(f"Класи: {class_names}")
print(f"Тренувальна вибірка: {len(train_subset)} зображень")
print(f"Валідаційна вибірка:  {len(val_subset)} зображень")
print(f"Тестова вибірка:       {len(test_dataset)} зображень")

images, labels = next(iter(train_loader))
print(f"\nФорма батчу зображень: {images.shape}")
print(f"Форма батчу міток:     {labels.shape}")

Шлях до тренувальних даних: datasets\seg_train\seg_train
Шлях до тестових даних:      datasets\seg_test\seg_test

Кількість класів: 6
Класи: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Тренувальна вибірка: 11227 зображень
Валідаційна вибірка:  2807 зображень
Тестова вибірка:       3000 зображень

Форма батчу зображень: torch.Size([32, 3, 150, 150])
Форма батчу міток:     torch.Size([32])


In [4]:
# 6. Визначення простої CNN моделі
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()

        self.features = nn.Sequential(
            # Блок 1
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Блок 2
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Блок 3
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Автоматичне зменшення до 1 × 1
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = SimpleCNN(num_classes=len(class_names)).to(device)

print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Line

In [5]:
# 7. Перевірка форми виходу моделі для одного батчу
images, labels = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)

print(f"Форма вхідного батчу:  {images.shape}")
print(f"Форма виходу моделі:   {outputs.shape}")

Форма вхідного батчу:  torch.Size([32, 3, 150, 150])
Форма виходу моделі:   torch.Size([32, 6])


In [6]:
# 8. Визначення функції втрат, оптимізатора та інших параметрів навчання

criterion = nn.CrossEntropyLoss()

# Оптимізатор Adam
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

# Автоматичне зменшення learning rate,
# якщо validation loss перестає покращуватися
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2
)

# Основні параметри майбутнього навчання
num_epochs = 10
best_val_loss = float('inf')
best_model_weights = None

print(f"Функція втрат: {criterion}")
print(f"Оптимізатор: {optimizer}")
print(f"Початковий learning rate: {optimizer.param_groups[0]['lr']}")
print(f"Кількість епох: {num_epochs}")

Функція втрат: CrossEntropyLoss()
Оптимізатор: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.0001
)
Початковий learning rate: 0.001
Кількість епох: 10


In [7]:
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = model(images)
loss = criterion(outputs, labels)

print(f"Форма outputs: {outputs.shape}")
print(f"Форма labels:  {labels.shape}")
print(f"Початкове значення loss: {loss.item():.4f}")

Форма outputs: torch.Size([32, 6])
Форма labels:  torch.Size([32])
Початкове значення loss: 1.7708


In [ ]:
# 9. Функції для навчання та валідації протягом однієї епохи

def train_one_epoch(model, loader, criterion, optimizer, device):
    """
    Навчання моделі протягом однієї епохи.

    Повертає:
        epoch_loss       — середнє значення функції втрат;
        epoch_accuracy   — точність;
        epoch_f1_macro   — F1-score macro;
        epoch_f1_weighted — F1-score weighted.
    """

    model.train()

    running_loss = 0.0
    total_samples = 0

    all_labels = []
    all_predictions = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        # Очищення градієнтів попереднього кроку
        optimizer.zero_grad(set_to_none=True)

        # Пряме поширення
        outputs = model(images)

        # Обчислення втрат
        loss = criterion(outputs, labels)

        # Зворотне поширення
        loss.backward()

        # Оновлення ваг моделі
        optimizer.step()

        # Накопичення втрат
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

        # Отримання прогнозованих класів
        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.detach().cpu().numpy())
        all_predictions.extend(predictions.detach().cpu().numpy())

    # Середня втрата за всю епоху
    epoch_loss = running_loss / total_samples

    # Перетворення списків у масиви NumPy
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)

    # Accuracy
    epoch_accuracy = np.mean(all_labels == all_predictions)

    # F1-score для багатокласової класифікації
    epoch_f1_macro = f1_score(
        all_labels,
        all_predictions,
        average='macro',
        zero_division=0
    )

    epoch_f1_weighted = f1_score(
        all_labels,
        all_predictions,
        average='weighted',
        zero_division=0
    )

    return (
        epoch_loss,
        epoch_accuracy,
        epoch_f1_macro,
        epoch_f1_weighted
    )


# 10. Валідація моделі протягом однієї епохи

def validate_one_epoch(model, loader, criterion, device):
    """
    Валідація моделі протягом однієї епохи.

    Ваги моделі під час валідації не оновлюються.
    """

    model.eval()

    running_loss = 0.0
    total_samples = 0

    all_labels = []
    all_predictions = []

    # Градієнти під час валідації не потрібні
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            # Пряме поширення
            outputs = model(images)

            # Обчислення втрат
            loss = criterion(outputs, labels)

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total_samples += batch_size

            # Отримання прогнозованих класів
            predictions = outputs.argmax(dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    # Середня втрата за всю валідаційну вибірку
    epoch_loss = running_loss / total_samples

    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)

    # Accuracy
    epoch_accuracy = np.mean(all_labels == all_predictions)

    # F1-score macro
    epoch_f1_macro = f1_score(
        all_labels,
        all_predictions,
        average='macro',
        zero_division=0
    )

    # F1-score weighted
    epoch_f1_weighted = f1_score(
        all_labels,
        all_predictions,
        average='weighted',
        zero_division=0
    )

    return (
        epoch_loss,
        epoch_accuracy,
        epoch_f1_macro,
        epoch_f1_weighted
    )

#11. Основний цикл навчання та валідації

# Історія навчання
history = {
    'train_loss': [],
    'train_accuracy': [],
    'train_f1_macro': [],
    'train_f1_weighted': [],

    'val_loss': [],
    'val_accuracy': [],
    'val_f1_macro': [],
    'val_f1_weighted': []
}


# Параметри навчання
num_epochs = 10
best_val_f1 = -float('inf')
best_model_weights = copy.deepcopy(model.state_dict())

start_time = time.time()


for epoch in range(num_epochs):
    epoch_start_time = time.time()

    # Навчання
    train_loss, train_accuracy, train_f1_macro, train_f1_weighted = (
        train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device
        )
    )

    # Валідація
    val_loss, val_accuracy, val_f1_macro, val_f1_weighted = (
        validate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device
        )
    )

    # Передаємо validation loss до планувальника
    scheduler.step(val_loss)

    # Збереження історії
    history['train_loss'].append(train_loss)
    history['train_accuracy'].append(train_accuracy)
    history['train_f1_macro'].append(train_f1_macro)
    history['train_f1_weighted'].append(train_f1_weighted)

    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)
    history['val_f1_macro'].append(val_f1_macro)
    history['val_f1_weighted'].append(val_f1_weighted)

    # Зберігаємо модель з найкращим validation F1
    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        best_model_weights = copy.deepcopy(model.state_dict())
        torch.save(
            best_model_weights,
            'best_simple_cnn_by_f1.pth'
        )
        best_marker = ' ← найкраща модель за F1'
    else:
        best_marker = ''

    current_lr = optimizer.param_groups[0]['lr']
    epoch_time = time.time() - epoch_start_time

    print(
        f"Епоха [{epoch + 1}/{num_epochs}] | "
        f"Час: {epoch_time:.1f} с | "
        f"LR: {current_lr:.6f}"
    )

    print(
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_accuracy:.4f} | "
        f"Train F1-macro: {train_f1_macro:.4f} | "
        f"Train F1-weighted: {train_f1_weighted:.4f}"
    )

    print(
        f"Val loss: {val_loss:.4f} | "
        f"Val acc: {val_accuracy:.4f} | "
        f"Val F1-macro: {val_f1_macro:.4f} | "
        f"Val F1-weighted: {val_f1_weighted:.4f}"
        f"{best_marker}\n"
    )


# Завантаження найкращих ваг після завершення навчання
model.load_state_dict(best_model_weights)
model.to(device)

total_time = time.time() - start_time

print(f"Загальний час навчання: {total_time / 60:.2f} хв")
print(f"Найкращий validation F1: {best_val_f1:.4f}")
print("Найкращі ваги моделі завантажено.")

Епоха [1/10] | Час: 451.5 с | LR: 0.001000
Train loss: 0.5550 | Train acc: 0.7985 | Train F1-macro: 0.7995 | Train F1-weighted: 0.7985
Val loss: 0.5022 | Val acc: 0.8101 | Val F1-macro: 0.8112 | Val F1-weighted: 0.8096 ← найкраща модель

Епоха [2/10] | Час: 397.4 с | LR: 0.001000
Train loss: 0.5383 | Train acc: 0.8044 | Train F1-macro: 0.8053 | Train F1-weighted: 0.8043
Val loss: 0.4697 | Val acc: 0.8326 | Val F1-macro: 0.8339 | Val F1-weighted: 0.8325 ← найкраща модель

Епоха [3/10] | Час: 392.2 с | LR: 0.001000
Train loss: 0.5365 | Train acc: 0.8069 | Train F1-macro: 0.8078 | Train F1-weighted: 0.8068
Val loss: 0.7985 | Val acc: 0.7143 | Val F1-macro: 0.7215 | Val F1-weighted: 0.7216

Епоха [4/10] | Час: 375.6 с | LR: 0.001000
Train loss: 0.5169 | Train acc: 0.8134 | Train F1-macro: 0.8142 | Train F1-weighted: 0.8132
Val loss: 0.4654 | Val acc: 0.8372 | Val F1-macro: 0.8389 | Val F1-weighted: 0.8375 ← найкраща модель

Епоха [5/10] | Час: 435.7 с | LR: 0.001000
Train loss: 0.5065 | Tr

Результати дуже хороші: проста CNN досягла найкращого валідаційного F1-macro = 0.8431 на 7-й епосі, а найвищу Val accuracy = 84.29% і Val F1-macro = 0.8447 — на 10-й епосі. За критерієм мінімальної Val loss збережено модель із 7-ї епохи (0.4582).


| Епоха | Train accuracy | Val accuracy | Val F1-macro | Val loss |
| ----- | -------------- | ------------ | ------------ | -------- |
| 1     | 79.85%         | 81.01%       | 0.8112       | 0.5022   |
| 2     | 80.44%         | 83.26%       | 0.8339       | 0.4697   |
| 4     | 81.34%         | 83.72%       | 0.8389       | 0.4654   |
| 7     | 82.63%         | 84.18%       | 0.8431       | 0.4582   |
| 10    | 83.01%         | 84.29%       | 0.8447       | 0.4614   |

Модель досягла цільового діапазону:

- орієнтовна accuracy: 75–85%;

- отримана найкраща validation accuracy: 84.29%;

- орієнтовний F1-score: 0.73–0.83;

- отриманий найкращий validation F1-score: 0.8447.

F1-score є гармонійним середнім між precision і recall, тому він інформативніший за одну лише accuracy для оцінювання якості по всіх класах.

